# Point (1-parameter) vs Gaussian (2-parameter) CNN ensemble: Cardinal Y1 complete

This is the first task of `CNN_photoz-ensemble-cardinal.ipynb` (noisy Cardinal, Y1 depth, i < 23, all 9 bands), with two versions of the ensemble trained on the same data:

- **1-parameter**: each network predicts a single redshift and is trained with MSE. The p(z) is a Gaussian whose width is the **spread between ensemble members**, which only reflects model uncertainty.
- **Gaussian (2-parameter)**: each network predicts a mean μ and a width σ per object and is trained with the Gaussian negative log-likelihood (`cnnpz.gaussian_nll`). The p(z) is an **equal-weight mixture** of the members' Gaussians, so it carries the per-object uncertainty each network learned (photometric noise, colour–redshift degeneracies) as well as the model spread.

Both are compared on the test set for point-estimate quality (bias, scatter, outliers) and p(z) calibration (PIT). Trained ensembles are saved under `MODEL_ROOT` and reloaded on re-runs, as in the original notebook.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from qp.metrics.pit import PIT

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

import cnnpz  # run from the repo root so this picks up the local source

## Configuration

In [ ]:
# Parametric paths (edit via environment variables, or defaults below)
CARDINAL_DATA_ROOT = os.environ.get("CNNPZ_CARDINAL_DATA_ROOT", "data")
FILTER_ROOT = os.environ.get("CNNPZ_FILTER_ROOT", "data")
MODEL_ROOT = os.environ.get("CNNPZ_MODEL_ROOT", "models/point_vs_gaussian")

TRAIN_FILE = os.path.join(CARDINAL_DATA_ROOT, "train_100k_noisy_y1_i23.parquet")
TEST_FILE = os.path.join(CARDINAL_DATA_ROOT, "test_100k_noisy_y1_i23.parquet")

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength. The Roman columns
# are read under their original names, so no renaming is needed.
BANDS = {
    "u": ("mag_u_lsst", os.path.join(FILTER_ROOT, "DC2LSST_u.res")),
    "g": ("mag_g_lsst", os.path.join(FILTER_ROOT, "DC2LSST_g.res")),
    "r": ("mag_r_lsst", os.path.join(FILTER_ROOT, "DC2LSST_r.res")),
    "i": ("mag_i_lsst", os.path.join(FILTER_ROOT, "DC2LSST_i.res")),
    "z": ("mag_z_lsst", os.path.join(FILTER_ROOT, "DC2LSST_z.res")),
    "y": ("mag_y_lsst", os.path.join(FILTER_ROOT, "DC2LSST_y.res")),
    "Y": ("Roman_obs_Y106", os.path.join(FILTER_ROOT, "roman_Y106.res")),
    "J": ("Roman_obs_J129", os.path.join(FILTER_ROOT, "roman_J129.res")),
    "H": ("Roman_obs_H158", os.path.join(FILTER_ROOT, "roman_H158.res")),
}
REF_BAND = "i"  # magnitudes are normalized by this band

N_SPLITS = 5  # ensemble members (K-fold), the same for both models
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)

MODELS = {"1-parameter": False, "Gaussian": True}  # name -> gaussian flag for train_ensembles

## Load training and test data

In [ ]:
training_y1 = cnnpz.read_catalog(TRAIN_FILE)
test_y1 = cnnpz.read_catalog(TEST_FILE)

X_y1, Y_y1 = cnnpz.catalog_to_XY(training_y1, BANDS, REF_BAND)
X_test_y1, Y_test_y1 = cnnpz.catalog_to_XY(test_y1, BANDS, REF_BAND)
mag_i_test = test_y1["mag_i_lsst"].to_numpy()
print("train:", X_y1.shape, "test:", X_test_y1.shape)

cnnpz.visualize_the_data(X_y1, Y_y1, np.arange(X_y1.shape[1]), title="Y1 training example")
plt.xlabel("wavelength bin")
plt.show()

## Train both ensembles (or load if already trained)

In [ ]:
trained, histories = {}, {}
for name, gaussian in MODELS.items():
    save_dir = os.path.join(MODEL_ROOT, f"y1_complete_{'gaussian' if gaussian else 'point'}_ensemble_CNN_6layers")
    if os.path.exists(os.path.join(save_dir, "norm_params.json")):
        trained[name] = cnnpz.load_ensemble(save_dir=save_dir)
        histories[name] = None
    else:
        trained[name], histories[name] = cnnpz.train_ensembles(
            cnnpz.build_model, X_y1, Y_y1, N_SPLITS=N_SPLITS, EPOCHS=EPOCHS, gaussian=gaussian
        )
        cnnpz.save_ensemble(trained[name], save_dir=save_dir)

Training curves. The losses are on different scales: MSE for the 1-parameter model, and Gaussian negative log-likelihood (which goes negative as σ shrinks) for the Gaussian model.

In [ ]:
if histories["1-parameter"] is not None:
    cnnpz.plot_ensemble_losses(histories["1-parameter"], ylim=(0, 0.001))
if histories["Gaussian"] is not None:
    cnnpz.plot_ensemble_losses(histories["Gaussian"], ylim=None)

## Predictions on the test set

`zmode` is the most probable redshift: the mean for the 1-parameter Gaussian, and the peak of the mixture for the Gaussian model. `width` is the standard deviation of each object's p(z).

In [ ]:
pz, z_point, width = {}, {}, {}
for name, models in trained.items():
    pz[name] = cnnpz.ensemble_predict(models, X_test_y1, ids=test_y1["galaxy_id"].to_numpy())
    z_point[name] = np.squeeze(pz[name].ancil["zmode"])
    width[name] = cnnpz.predictions_mean_std(pz[name])[1].ravel()
    print(f"{name}: median p(z) width = {np.median(width[name]):.4f}")

## Point estimates

Δz = (z_phot − z_true) / (1 + z_true), using `zmode` as z_phot.

In [ ]:
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)

stats = {}
for name in trained:
    stats[name] = cnnpz.get_all_stats(
        Y_test_y1, z_point[name], mag_i_test, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
    )
    cnnpz.plot_stats(*stats[name], Y_test_y1, z_point[name], redshift_bins, imag_bins, mag_i_test)
    plt.suptitle(name, y=1.02)
    plt.show()

Side by side: solid lines are the 1-parameter model, dashed lines the Gaussian model.

In [ ]:
names = list(trained)
display(Markdown(cnnpz.stats_to_markdown(stats[names[0]][0], stats[names[1]][0], data_title=names)))
cnnpz.compare_binned_stats(
    redshift_bins, imag_bins, stats[names[0]][1], stats[names[0]][2], stats[names[1]][1], stats[names[1]][2]
)
plt.show()

## p(z) calibration: PIT

PIT = CDF of each object's p(z) at its true redshift. For calibrated p(z) the PIT values are uniform: a flat histogram and a Q–Q curve on the diagonal. A U shape means p(z) too narrow, a hump means too wide. KS, CvM and Anderson–Darling measure the distance from uniform (lower is better). `outlier` is the fraction of PIT values in the extreme tails (< 0.0001 or > 0.9999).

In [ ]:
pit_values, pit_stats = {}, {}
for name in trained:
    pit = PIT(pz[name], Y_test_y1)
    pit_values[name] = pit.pit_samps
    pit_stats[name] = {
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        "outlier": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
    }

fig, axarr = plt.subplots(1, 2, figsize=[13, 5])
q = np.linspace(0, 1, 101)
for name in trained:
    axarr[0].hist(pit_values[name], bins=50, range=(0, 1), density=True, histtype="step", lw=2, label=name)
    axarr[1].plot(q, np.quantile(pit_values[name], q), lw=2, label=name)
axarr[0].axhline(1, color="k", ls="--")
axarr[0].set_xlabel("PIT")
axarr[0].set_ylabel("density")
axarr[0].legend()
axarr[1].plot([0, 1], [0, 1], "k--")
axarr[1].set_xlabel("uniform quantile")
axarr[1].set_ylabel("PIT quantile")
plt.tight_layout()
plt.show()

pd.DataFrame(pit_stats).round(4)

## Do the widths match the actual errors?

Left and middle: median p(z) width / (1 + z) in bins of i magnitude and true redshift, against the actual scatter of Δz (biweight, from `get_all_stats`). For calibrated p(z) the two should track each other. Right: residuals divided by the predicted width, which should follow a unit Gaussian.

In [ ]:
def binned_median(x, values, bins):
    idx = np.digitize(x, bins) - 1
    return np.array([np.median(values[idx == i]) if np.any(idx == i) else np.nan for i in range(len(bins) - 1)])


fig, axarr = plt.subplots(1, 3, figsize=[18, 5])
for k, name in enumerate(trained):
    rel_width = width[name] / (1 + Y_test_y1)
    for ax, x, bins, binned in [
        (axarr[0], mag_i_test, imag_bins, stats[name][2]),
        (axarr[1], Y_test_y1, redshift_bins, stats[name][1]),
    ]:
        centres = (bins[1:] + bins[:-1]) / 2
        ax.plot(centres, binned_median(x, rel_width, bins), "-o", color=f"C{k}", label=f"{name}: median width")
        ax.plot(centres, [s[2] for s in binned], "--", color=f"C{k}", label=f"{name}: actual scatter")
    axarr[2].hist(
        (z_point[name] - Y_test_y1) / width[name], bins=np.linspace(-6, 6, 121), density=True, histtype="step", lw=2, label=name
    )

zz = np.linspace(-6, 6, 200)
axarr[2].plot(zz, np.exp(-0.5 * zz**2) / np.sqrt(2 * np.pi), "k--", label="unit Gaussian")
axarr[0].set_xlabel("i magnitude")
axarr[1].set_xlabel("true redshift")
for ax in axarr[:2]:
    ax.set_ylabel(r"$\sigma / (1+z)$")
    ax.set_yscale("log")
    ax.legend(fontsize=9)
axarr[2].set_xlabel("(z_mode - z_true) / width")
axarr[2].legend(fontsize=10)
plt.tight_layout()
plt.show()

## Example p(z)

A few random test objects: each model's p(z), with the true redshift as a dashed line.

In [ ]:
rng = np.random.default_rng(1)
examples = rng.choice(len(Y_test_y1), size=8, replace=False)
z_grid = np.linspace(0, 3, 601)

fig, axarr = plt.subplots(2, 4, figsize=[18, 7], sharex=True)
for ax, i in zip(axarr.ravel(), examples):
    for k, name in enumerate(trained):
        ax.plot(z_grid, np.squeeze(pz[name][int(i)].pdf(z_grid)), color=f"C{k}", label=name)
    ax.axvline(Y_test_y1[i], color="k", ls="--")
    ax.set_xlim(max(0, Y_test_y1[i] - 0.6), Y_test_y1[i] + 0.6)
    ax.set_title(f"i = {mag_i_test[i]:.1f}, z = {Y_test_y1[i]:.2f}", fontsize=11)
axarr[0, 0].legend(fontsize=9)
for ax in axarr[1]:
    ax.set_xlabel("z")
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = {}
for name in trained:
    bias, bias_err, scatter, outlier_rate, abs_outlier_rate = stats[name][0]
    summary[name] = {
        "bias": bias,
        "scatter": scatter,
        "outlier rate (3 sigma)": outlier_rate,
        "outlier rate (|dz| > 0.2)": abs_outlier_rate,
        "median p(z) width": np.median(width[name]),
        **{f"PIT {k}": v for k, v in pit_stats[name].items()},
    }
pd.DataFrame(summary).round(4)